# **Modelado Predictivo del Nivel del Río en la Estación Hidrológica "El Banco - Aut" Mediante Aprendizaje Automático Supervisado**

# **1. Descripción del problema e inspección del conjunto de datos**
# **1.1. Descripción del problema**

Este proyecto aborda un problema de predicción de series de tiempo mediante aprendizaje automático supervisado, enfocado específicamente en pronosticar el nivel del río Magdalena en la estación automática "EL BANCO - AUT" (Código: 25027020).

El conjunto de datos oficial, obtenido a través del portal: "https://www.datos.gov.co/Ambiente-y-Desarrollo-Sostenible/Nivel-Instant-neo-del-Rio/bdmn-sqnh/about_data", contiene originalmente 12 variables. A continuación se muestra una descripción de las variables:
*   CodigoEstacion: Identificación de la estación dentro del catálogo de estaciones.
*   CodigoSensor: Código de identificación asignado al sensor.
*   FechaObservación: Fecha en la cual se realiza la medición.
*   ValorObservado: Valor medido.
*   NombreEstacion: Identificación de la estación dentro del catálogo de estaciones.
*   Departamento: Nombre del departamento donde se ubica la estación.
*   Municipio: Nombre del Municipio donde se localiza la estación.
*   ZonaHidrografica: Zona hidrográfica sobre la cual está ubicada la estación.
*   Latitud: Latitud en la cual se ubica la estación.
*   Longitud: Longitud en la cual se ubica la estación.
*   DescripcionSensor: Nivel instantáneo.
*   UnidadMedida: Metros.

Dado que el estudio se enfoca en el comportamiento histórico y predictivo de una estación hidrológica, el alcance del preprocesamiento se delimita únicamente a las variables  *FechaObservada* y *ValorObservado*, descartando las demás variables que no aportan valor directo al modelo.



# **1.2. Importación de librerías**
Se utilizan librerías como pandas y seaborn para la inspección y exploración de los datos.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# **1.3. Carga del dataset**
Se toman únicamente las dos variables de interés para nuestro análisis.

In [2]:
columnas_deseadas = ["FechaObservacion", "ValorObservado"]
df = pd.read_csv("Nivel_Instantáneo_del_Rio_20261020.csv", usecols=columnas_deseadas)

df.info()
display(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 315876 entries, 0 to 315875
Data columns (total 2 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   FechaObservacion  315876 non-null  object
 1   ValorObservado    315876 non-null  object
dtypes: object(2)
memory usage: 4.8+ MB


,FechaObservacion,ValorObservado
0,2011 Aug 15 11:00:00 AM,"6,73"
1,2005 Nov 15 02:00:00 PM,"6,15"
2,2011 May 10 05:00:00 AM,"9,43"
3,2009 Apr 29 07:00:00 AM,"7,25"
4,2016 Sep 17 07:00:00 PM,"6,12"


Las variables *FechaObservada* y *ValorObservado* son de tipo object, se procede a hacer el cambio a tipo datetime y float respectivamente.

In [3]:
df["FechaObservacion"] = pd.to_datetime(df["FechaObservacion"], errors="coerce")

df["ValorObservado"] = df["ValorObservado"].astype(str).str.replace(",", ".", regex=False)
df["ValorObservado"] = pd.to_numeric(df["ValorObservado"], errors="coerce")

df.info()
display(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 315876 entries, 0 to 315875
Data columns (total 2 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   FechaObservacion  315876 non-null  datetime64[ns]
 1   ValorObservado    315876 non-null  float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 4.8 MB


,FechaObservacion,ValorObservado
0,2011-08-15 11:00:00,6.73
1,2005-11-15 14:00:00,6.15
2,2011-05-10 05:00:00,9.43
3,2009-04-29 07:00:00,7.25
4,2016-09-17 19:00:00,6.12


Se ordenan cronológicamente los registros.

In [4]:
df = df.sort_values(by="FechaObservacion", ascending=True).reset_index(drop=True)

display(df.head())
display(df.tail())

,FechaObservacion,ValorObservado
0,2005-01-01 17:00:00,8.65
1,2005-01-01 18:00:00,8.65
2,2005-01-01 19:00:00,8.66
3,2005-01-01 20:00:00,8.69
4,2005-01-01 21:00:00,8.67


,FechaObservacion,ValorObservado
315871,2026-10-19 19:00:00,5.05
315872,2026-10-19 20:00:00,5.06
315873,2026-10-19 21:00:00,5.06
315874,2026-10-19 22:00:00,5.08
315875,2026-10-19 23:00:00,5.09


# **1.4. Inspección inicial**
Se revisan valores faltantes, fechas duplicadas, posibles registros duplicados y valores
numéricos atípicos o imposibles.

In [5]:
print("\nResumen de valores faltantes:")
print(df.isna().sum())
df_fechas_duplicadas = df[df["FechaObservacion"].duplicated()].copy()
print("\nFechas duplicadas:", df["FechaObservacion"].duplicated().sum())
display(df_fechas_duplicadas.head())
print("\nRegistros duplicados:", df.duplicated().sum())

print("\nResumen de variable 'ValorObservado':")
display(df.select_dtypes(include="number").describe().T)

# Calcular Q1, Q3 e IQR
Q1 = df["ValorObservado"].quantile(0.25)
Q3 = df["ValorObservado"].quantile(0.75)
IQR = Q3 - Q1

# Definir los límites superior e inferior
limite_inferior = Q1 - (1.5 * IQR)
limite_superior = Q3 + (1.5 * IQR)

# Filtrar los valores atípicos
outliers = df[(df["ValorObservado"] < limite_inferior) | (df["ValorObservado"] > limite_superior)]

print(f"\nCantidad de valores atípicos encontrados: {len(outliers)}")
display(outliers.head())


Resumen de valores faltantes:
FechaObservacion    0
ValorObservado      0
dtype: int64

Fechas duplicadas: 18054


,FechaObservacion,ValorObservado
102491,2019-08-28 00:00:00,6.39
102492,2019-08-28 00:00:00,6.39
102494,2019-08-28 01:00:00,6.39
102495,2019-08-28 01:00:00,6.39
102497,2019-08-28 02:00:00,6.39



Registros duplicados: 18054

Resumen de variable 'ValorObservado':


,count,mean,std,min,25%,50%,75%,max
ValorObservado,315876.0,6.538216,1.397943,0.0,5.77,6.47,7.29,172.42



Cantidad de valores atípicos encontrados: 926


,FechaObservacion,ValorObservado
2572,2005-04-22 17:00:00,20.49
2573,2005-04-22 19:00:00,167.67
4683,2005-07-22 03:00:00,172.42
18578,2007-02-22 13:00:00,2.40
18579,2007-02-22 14:00:00,2.40


Se puede observar que no hay ningún valor faltante, pero si hay 18054 registros duplicados por lo que pudimos evidenciar se trata de un error del sensor al momento de enviar la información.

Utilizando el método del rango intercuartílico (IQR), se evaluó la distribución de la variable *ValorObservado*. Este análisis arrojó un tota de 926 valores atípicos ubicados en el extremo superior de la distribución.

Procedemos a eliminar estos datos duplicados, dejando un único registro.

In [6]:
df = df.drop_duplicates().reset_index(drop=True)

#Verificamos duplicados restantes
print("\nFechas duplicadas restantes:", df["FechaObservacion"].duplicated().sum())
print("\nRegistros duplicados restantes:", df.duplicated().sum())



Fechas duplicadas restantes: 0

Registros duplicados restantes: 0


Una vez eliminados los registros duplicados, procedemos a analizar la consecutividad de los valores atípicos para luego decidir el tratamiento (como interpolación, eliminación o imputación) a reailzar sobre dichos valores.

In [16]:
# Aseguramos que el DataFrame esté ordenado cronológicamente
outliers = outliers.sort_values(by="FechaObservacion") 

# Se calcula la diferencia de tiempo con respecto a la fila anterior
outliers["diferencia_tiempo"] = outliers["FechaObservacion"].diff()

print("--- Frecuencias más comunes entre registros atípicos ---\n")
display(outliers["diferencia_tiempo"].value_counts().head(20))

--- Frecuencias más comunes entre registros atípicos ---



diferencia_tiempo
0 days 00:10:00     444
0 days 01:00:00     263
0 days 00:20:00      41
0 days 00:30:00      16
0 days 02:00:00      14
0 days 00:40:00      12
0 days 03:00:00       7
0 days 00:00:00       7
0 days 00:50:00       6
0 days 08:00:00       3
0 days 01:20:00       3
0 days 01:10:00       3
0 days 21:00:00       2
1 days 00:40:00       2
0 days 04:00:00       2
0 days 15:00:00       2
0 days 23:30:00       2
0 days 17:50:00       2
90 days 08:00:00      2
0 days 05:00:00       2
Name: count, dtype: int64

Tras analizar la diferencia de tiempo entre registros atípicos, se evidenció que el intervalo más frecuente es de 10 minutos, lo que indica que no corresponden a anomalías aisladas, sino a bloques de datos consecutivos. En consecuencia, se procede a cuantificar la longitud de dichos bloques para fundamentar su posterior tratamiento.

In [8]:
# Marcamos con True cuando la diferencia sea mayor a la frecuencia base (mayor a 10 min). 
outliers["nuevo_bloque"] = (outliers["diferencia_tiempo"] > pd.Timedelta(minutes=10)).fillna(True)

# Creamos un identificador único para cada racha acumulando los True con .cumsum()
outliers["id_racha"] = outliers["nuevo_bloque"].cumsum()

# Agrupamos por ese ID y contamos cuántos registros tiene cada bloque
longitudes_de_bloques = outliers.groupby("id_racha").size()

print("\n--- Estadísticas de los bloques ---")
print(longitudes_de_bloques.describe())


--- Estadísticas de los bloques ---
count    475.000000
mean       1.949474
std        3.540241
min        1.000000
25%        1.000000
50%        1.000000
75%        1.000000
max       43.000000
dtype: float64


Este análisis demostró que el 75% de los registros atípicos se presentan en bloques muy cortos, con un promedio general de 1.95 registros por bloque. Se identifcó un bloque máximo con 43 registros consecutivos.

In [20]:
# Se usan los índices del DataFrame de outliers para convertir esos valores específicos en NaN en el df principal
df.loc[outliers.index, "ValorObservado"] = np.nan

print(f"Valores convertidos a NaN usando los índices del df_outliers: {df['ValorObservado'].isna().sum()}")

# Se aplica interpolación lineal limitada (máximo 3 consecutivos)
df["ValorObservado"] = df["ValorObservado"].interpolate(method="linear", limit=4, limit_direction="both")

print(f"Nulos restantes tras la interpolación: {df['ValorObservado'].isna().sum()}")

outliers_nulos = outliers[outliers['FechaObservacion'].isin(df[df['ValorObservado'].isna()]['FechaObservacion'])] 
outliers_nulos = outliers_nulos.sort_values(by="FechaObservacion") 
outliers_nulos['diferencia_tiempo'] = outliers_nulos['FechaObservacion'].diff()
print(outliers_nulos['diferencia_tiempo'].value_counts().head(10))

# Marcamos con True cuando la diferencia sea mayor a la frecuencia base (mayor a 10 min). 
outliers_nulos["nuevo_bloque"] = (outliers_nulos["diferencia_tiempo"] > pd.Timedelta(hours=1)).fillna(True)

# Creamos un identificador único para cada racha acumulando los True con .cumsum()
outliers_nulos["id_racha"] = outliers_nulos["nuevo_bloque"].cumsum()

# Agrupamos por ese ID y contamos cuántos registros tiene cada bloque
longitud = outliers_nulos.groupby("id_racha").size()

print("\n--- Estadísticas de los bloques ---")
print(longitud.describe())

Valores convertidos a NaN usando los índices del df_outliers: 926
Nulos restantes tras la interpolación: 420
diferencia_tiempo
0 days 01:00:00       160
0 days 00:10:00         7
729 days 00:00:00       1
798 days 09:00:00       1
2692 days 10:00:00      1
1491 days 09:00:00      1
5 days 12:30:00         1
Name: count, dtype: int64

--- Estadísticas de los bloques ---
count      6.000000
mean      28.833333
std       56.513420
min        2.000000
25%        3.250000
50%        7.500000
75%        9.500000
max      144.000000
dtype: float64


Los resultados indican que la mayoría de los registros cumplen estrictamente con la periocidad nominal de una hora.

# **1.5. Agrupación por fecha**
Se evalúa cuántos registros aporta el sensor por día.

In [10]:
# Agrupamos por fecha (sin la hora) y contamos cuántas mediciones hubo cada día
observaciones_por_dia = (df["FechaObservacion"].dt.date.value_counts().sort_index())

print("--- Resumen estadístico de observaciones por día ---")
display(observaciones_por_dia.describe())

# Visualizamos los primeros días para ver si hay días con más registros de lo normal
print("\n--- Conteo de registros en los primeros días ---")
display(observaciones_por_dia.head(5))

--- Resumen estadístico de observaciones por día ---


count    6259.00000
mean       47.58300
std        47.39327
min         1.00000
25%        24.00000
50%        24.00000
75%        24.00000
max       146.00000
Name: count, dtype: float64


--- Conteo de registros en los primeros días ---


FechaObservacion
2005-01-01     7
2005-01-02    24
2005-01-03    24
2005-01-04    24
2005-01-05    24
Name: count, dtype: int64

El análisis confirma que el sensor mantiene una cobertura robusta y constante con una mediana de 24 registros diarios. A pesar de haber identificado días con con una menor cantidad de mediciones (con un mínimo de 13 registros diarios), el comportamiento general refleja una alta confiabilidad operativa en la ventana temporal estudiada.

In [11]:
# Filtramos saltos estrictamente mayores a 3 horas
umbral_gap = pd.Timedelta(hours=3)
periodos_sin_info = df[df["diferencia_tiempo"] > umbral_gap]

print(f"Se encontraron {len(periodos_sin_info)} periodos sin información:")
display(periodos_sin_info[["fechaobservacion", "diferencia_tiempo"]])

KeyError: 'diferencia_tiempo'

Al filtrar los intervalos de tiempo superiores a 3 horas, se identificaron únicamente 10 interrupciones en todo el periodo de estudio. La gran mayoría de estos eventos corresponden a retrasos operativos menores de entre 4 y 11 horas. El único vacío de información significativo se registró el 20 de agosto de 2026 con una duración de 2 días y 2 horas. Esto evidencia que, fuera de eventos técnicos aislados, la estación presenta una continuidad robusta y confiable para la serie temporal.

Se crea un dataframe temporal, donde la variable temporal se convierte en el índice del dataframe, permitiendo aplicar operaciones de remuestreo para calcular tanto el promedio diario como semanal del nivel del río.

In [ ]:
# Asegurarnos de que el DataFrame tiene la fecha como índice temporal
df_temp = df.copy()
df_temp["fechaobservacion"] = pd.to_datetime(df_temp["fechaobservacion"])
df_temp = df_temp.set_index("fechaobservacion")

# Ordenamos el índice temporal por seguridad
df_temp = df_temp.sort_index()

# Agregación diaria: promedio del nivel por día ('D')
df_diario = df_temp.resample("D")["valorobservado"].mean()

# Agregación semanal: promedio del nivel por semana ('W')
df_semanal = df_temp.resample("W")["valorobservado"].mean()

# Comparar cuántos puntos (filas) produce cada alternativa
print(f"Puntos generados por la agregación DIARIA: {len(df_diario)}")
print(f"Puntos generados por la agregación SEMANAL: {len(df_semanal)}")

# Primeras filas de cada uno para contrastar
print("\n--- Primeros registros Agregación Diaria ---")
display(df_diario.head())

print("\n--- Primeros registros Agregación Semanal ---")
display(df_semanal.head())

# **1.6. Visualización de la serie temporal**

In [ ]:

sns.set_theme(style="whitegrid")
plt.figure(figsize=(14, 6))

# Graficar la serie diaria de fondo (más delgada y traslúcida)
plt.plot(
    df_diario.index,
    df_diario.values,
    color="#aec7e8",
    linewidth=1.2,
    label="Promedio Diario (Detalle)",
)

# Graficar la serie semanal encima (más gruesa y destacada)
plt.plot(
    df_semanal.index,
    df_semanal.values,
    color="#1f77b4",
    linewidth=2.5,
    marker="o",
    markersize=5,
    label="Promedio Semanal (Variable Objetivo)",
)

# Títulos y formato
plt.title(
    (
        "Comparativa de Agregación Temporal: Promedio Diario vs. Promedio Semanal"
        " - Estación El Banco"
    ),
    fontsize=14,
    fontweight="bold",
    pad=15,
)
plt.xlabel("Fecha", fontsize=12, labelpad=10)
plt.ylabel("Nivel Observado (m)", fontsize=12, labelpad=10)
plt.xticks(rotation=45)
plt.legend(loc="upper left", fontsize=11)
plt.tight_layout()

plt.show()

La gráfica permite contrastar la fluctuación detallada del promedio diario frente al comportamiento suavizado del promedio semanal.

**Análisis visual:**

La curva azul claro (promedio diario) evidencia la alta frecuencia de las mediciones y la presencia de los picos correspondientes a las ondas de avenida a finales de septiembre e inicios de octubre.

La línea azul oscura (promedio semanal) filtra eficazmente el ruido telemático de alta frecuencia y traza con claridad la tendencia macro del nivel del río. Esta suavización confirma que el modelado a escala semanal es la estrategia correcta para capturar la dinámica hidrológica sin las distorsiones de las variaciones intra-diarias